In [1]:
from main import sheet_processor
import logging
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json
import os
from utils.processing import DataSampler, create_report, make_json_safe
import traceback

# How to set log level to debug
logging.basicConfig(level=logging.DEBUG)

logger = logging.getLogger(__name__)

## Model Selection

In this section, we specify which models the system will use. To simplify the initial setup and testing workflow, we currently use a single selected model across all processing components: **PII Detection, PII Reflection, Non-PII Detection, and ReadMe Detection**. This approach allows us to validate the end-to-end pipeline without introducing variability from multiple model behaviors.

At the moment, only two models are deployed through the Azure service: **GPT-4.1 Mini** and **GPT-4.1 Nano**. These models are the available choices for experimentation and integration. As the project evolves, additional models can be added or different models can be assigned to specific components for more specialized behavior.


In [2]:
MODEL = 'gpt-5-mini'

ALLOWED_MODELS = ['gpt-4.1-mini', 'gpt-4.1-nano', 'gpt-5-nano', 'gpt-5-mini']

if MODEL not in ALLOWED_MODELS:
    raise ValueError(f'Invalid model: {MODEL}. Please use one of the following: {", ".join(ALLOWED_MODELS)}')

# Dataset Selection

You can select a dataset in one of two ways:

1. **Using a download URL**: Provide the URL to a CSV, XLS, or XLSX file.
2. **Using a local file**: Choose a file from the `research/data` folder in the project.

This flexibility allows you to work with both online datasets and local files for testing and analysis.


In [3]:
file_path = 'research/data/'  # This is a local test file

# DataSampler

The `DataSampler` class is used to extract a subset of rows from the original dataset.  
This sampling helps to:

- **Reduce memory usage** when working with large datasets.
- **Increase processing speed** during classification and analysis.

By working on a smaller, representative portion of the data, we can efficiently test the pipeline and classifiers without loading the entire dataset.


In [4]:
sampler = DataSampler()

# Ground truth

This part of the code makes the sdd reports, but empty so we can later on fill in the real values


In [5]:
# # Start by making empty reports for each file as groundtruth2 to fill in later

# sampler = DataSampler()

# with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
#     isp = json.load(f)

# isp = isp['default']

# for file in os.listdir('research/data'):
#     file_path = f'research/data/{file}'
#     # If file already exists, skip
#     if os.path.exists(f'research/results/test_results/groundtruth2/{file}.json'):
#         logger.info(f'File {file} already exists. Skipping.')
#         continue
#     try:
#         sdd_report = create_report(file_path)
#         sdd_report = make_json_safe(sdd_report)
#         with open(f'research/results/test_results/groundtruth2/{file}.json', 'w') as f:
#             json.dump(sdd_report, f, indent=4)
#         logger.info(f'Report saved to research/results/test_results/groundtruth2/{file}.json')
#     except Exception as e:
#         logger.warning(f'Error processing file: {file} {e}')
#         logger.info(f'Error details: {traceback.format_exc()}')
#         break
#         continue

# Processing Each Sheet Individually

For each sheet in the dataset, we perform the following processing steps:

1. **PII Detection** – Identify columns containing personally identifiable information.
2. **PII Reflection Detection** – Detect columns that might indirectly reveal PII.
3. **Non-PII Detection** – Classify remaining columns that do not contain sensitive information.

**Special Case:**  
If a sheet is named `readme`, `instructions`, or `metadata`, we skip the column-level classification and instead perform a **simple ReadMe scan** to extract relevant information from the documentation.


In [9]:
# Start making sdd reports for a model
MODEL = 'gpt-4.1-mini'
# Start by making empty reports for each file as groundtruth2 to fill in later

# Make a folder for the model
os.makedirs(f'research/results/test_results/{MODEL}', exist_ok=True)

sampler = DataSampler()

with open('/Users/liangtelkamp/Documents/GitHub/hdx-ssd-pipeline/data/isps.json', 'r') as f:
    isp = json.load(f)

isp = isp['default']

for file in os.listdir('research/data'):
    file_path = f'research/data/{file}'
    # Check if file already exists
    output_path = f'research/results/test_results/{MODEL}/{file}.json'
    if os.path.exists(output_path):
        logger.info(f'File {file} already exists, loading existing report')
        # Load the existing report
        with open(output_path, 'r') as f:
            sdd_report = json.load(f)
        logger.info(f'Report loaded from {output_path}')
    else:
        # Check if file is already processed
        if not os.path.exists(f'research/results/test_results/groundtruth2/{file}.json'):
            logger.warning(f'File {file} not found in groundtruth2, skipping')
            continue
        else:
            with open(f'research/results/test_results/groundtruth2/{file}.json', 'r') as f:
                logger.info(f'File {file} found in groundtruth2, loading')
                sdd_report = json.load(f)

    reports = []
    for sheet in sdd_report:
        sdd_report = sheet_processor(sheet, isp, MODEL)
        reports.append(sdd_report)
    with open(output_path, 'w') as f:
        json.dump(reports, f, indent=4)
    logger.info(f'Report saved to {output_path}')

[70359 - 8687722688] 2026-01-15 12:24:06,373 WARNI [__main__:28] File bgd_dataset_joint-msna_refugee_september-2019.xlsx not found in groundtruth2, skipping
[70359 - 8687722688] 2026-01-15 12:24:06,374 INFO  [__main__:32] File afghanistan_june_protection.csv found in groundtruth2, loading


Reflecting on PII sensitivity: 100%|██████████| 46/46 [00:03<00:00, 12.30it/s]


[70359 - 8687722688] 2026-01-15 12:24:35,873 INFO  [__main__:41] Report saved to research/results/test_results/gpt-4.1-mini/afghanistan_june_protection.csv.json
[70359 - 8687722688] 2026-01-15 12:24:35,875 INFO  [__main__:32] File ujana-coffee-project-database-2019-2024.xlsx found in groundtruth2, loading


Reflecting on PII sensitivity: 100%|██████████| 2/2 [00:00<00:00,  3.07it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/ujana-coffee-project-database-2019-2024.xlsx', 'file_url': None, 'sheet_name': 'Farmers Trained', 'processing_timestamp': '2026-01-03 18:13:48', 'processing_success': True, 'n_records': 998, 'n_columns': 14, 'completion_tokens': 266, 'prompt_tokens': 1518, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': '', 'sample_values': ['27/09/2024', 'Kamuli', 'Town Council', 'AFOVA/MAHIO', 'Female'], 'pii': {'entity_type': 'GENDER', 'sensitive': True}}, {'column_name': 'Skills trained', 'sample_values': ['Baking cakes ', 'African earrings ', 'African handmade sandals ', 'Mushroom growing ', 'Notebooks'], 'pii': {'entity_type': 'None', 'sensitive': False}}], 'pii_classifier_model': 'gpt-4.1-mini', 'pii_reflection_model': 'gpt-4.1-mini', 'non_pii_model': 'gpt-4.1-mini', 'non_pii': {'sensitivity': 'NON_SENSITIVE', 'sensitive_columns': [], 'cited_isp_rules': ["NON_SENSITIVE: ['Humanitarian Needs Ov

...

Non-PII classification: {'resource_id': None, 'file_name': 'research/data/afghanistan_june_food.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2026-01-03 18:21:02', 'processing_success': True, 'n_records': 199, 'n_columns': 42, 'completion_tokens': 442, 'prompt_tokens': 29810, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'observed_time', 'sample_values': ['2025-06-03T07:38:39.507Z', '2025-06-01T13:04:12.381Z', '2025-06-03T10:27:47.558Z', '2025-06-01T13:29:07.920Z', '2025-06-01T13:12:56.969Z'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_version', 'sample_values': ['8', '8', '8', '8', '8'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_name', 'sample_values': ['Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community', 'Food Availability in Your Community'], 'pii': {'entity_t

Reflecting on PII sensitivity: 100%|██████████| 37/37 [00:03<00:00,  9.63it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/2023-the-effects-on-health-care-of-the-use-of-explosive-weapons-data (1).xlsx', 'file_url': None, 'sheet_name': '2023 Data', 'processing_timestamp': '2026-01-03 18:21:03', 'processing_success': True, 'n_records': 806, 'n_columns': 37, 'completion_tokens': 376, 'prompt_tokens': 22334, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'Date', 'sample_values': ['2023-10-07T00:00:00', '2023-04-18T00:00:00', '2023-06-28T00:00:00', '2023-10-18T00:00:00', '2023-06-27T00:00:00'], 'pii': {'entity_type': 'BIRTH_DATE', 'sensitive': True}}, {'column_name': 'Event Description', 'sample_values': ['October 2023: A male volunteer with an LNGO was held hostage, along with his wife and 12 other civilians by Hamas militants. He was subsequently killed during a rescue operation by Israeli forces (coded separately in event ID45130).', 'April 2023: A doctor, a nurse, patients and patient attendants were arr

Reflecting on PII sensitivity: 100%|██████████| 43/43 [00:03<00:00, 11.01it/s]


Non-PII classification: {'resource_id': None, 'file_name': 'research/data/afghanistan_june_education.csv', 'file_url': None, 'sheet_name': 'sheet1', 'processing_timestamp': '2026-01-03 18:21:03', 'processing_success': True, 'n_records': 199, 'n_columns': 43, 'completion_tokens': 393, 'prompt_tokens': 29679, 'pii_sensitive': True, 'non_pii_sensitive': True, 'columns': [{'column_name': 'observed_time', 'sample_values': ['2025-06-02T04:49:53.153Z', '2025-06-03T02:18:19.884Z', '2025-06-02T17:29:21.108Z', '2025-06-02T16:06:22.049Z', '2025-06-02T17:12:51.830Z'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_version', 'sample_values': ['10', '10', '10', '10', '10'], 'pii': {'entity_type': 'None', 'sensitive': False}}, {'column_name': 'form_name', 'sample_values': ['Education in Your Community', 'Education in Your Community', 'Education in Your Community', 'Education in Your Community', 'Education in Your Community'], 'pii': {'entity_type': 'None', 'sensitive': Fal

# Save


# Evaluation


In [7]:
def compare_pii_columns(gt_reports, pred_reports):
    """Compare PII sensitivity for all columns across sheets."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        gt_cols = {c['column_name']: c['pii']['sensitive'] for c in gt['columns']}
        pred_cols = {c['column_name']: c['pii']['sensitive'] for c in pred['columns']}
        for col_name in gt_cols:
            records.append({'column_name': col_name, 'true': gt_cols[col_name], 'pred': pred_cols.get(col_name, False)})
    return pd.DataFrame(records)


def compare_pii_table_level(gt_reports, pred_reports):
    """Compare PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('pii_sensitive', False), 'pred': pred.get('pii_sensitive', False)})
    return pd.DataFrame(records)


def compare_non_pii_table_level(gt_reports, pred_reports):
    """Compare non-PII sensitivity at the table level."""
    records = []
    for gt, pred in zip(gt_reports, pred_reports):
        records.append({'true': gt.get('non_pii_sensitive', False), 'pred': pred.get('non_pii_sensitive', False)})
    return pd.DataFrame(records)


def calculate_metrics(df: pd.DataFrame):
    """Compute accuracy, precision, recall, and F1 score."""
    return {
        'accuracy': accuracy_score(df['true'], df['pred']),
        'precision': precision_score(df['true'], df['pred'], zero_division=0),
        'recall': recall_score(df['true'], df['pred'], zero_division=0),
        'f1': f1_score(df['true'], df['pred'], zero_division=0),
    }

In [8]:
filename = 'data.xlsx'

# Read the groundtruth2 and predictions
with open(f'research/results/test_results/groundtruth2/{filename}.json', 'r') as f:
    groundtruth2 = json.load(f)
with open(f'research/results/test_results/gpt-5-nano/{filename}.json', 'r') as f:
    predictions = json.load(f)

# Calculate metrics
metrics = {
    filename: {
        'pii_columns': calculate_metrics(compare_pii_columns(groundtruth2, predictions)),
        'pii_table_level': calculate_metrics(compare_pii_table_level(groundtruth2, predictions)),
        'non_pii_table_level': calculate_metrics(compare_non_pii_table_level(groundtruth2, predictions)),
    }
}
metrics

FileNotFoundError: [Errno 2] No such file or directory: 'research/results/test_results/groundtruth2/data.xlsx.json'